# M10 — EDA: English popularity vs. Czech bestseller status

**Question (from supervisor):** Are bestsellers on the English market (books with many Goodreads ratings) automatically bestsellers in Czech, and vice versa — are there Czech bestsellers that were *not* popular in their original language?

**Why this matters for the thesis:** If foreign popularity strongly predicts Czech bestseller status, our model is redundant — a publisher could just sort foreign books by ratings count and pick the top. If the relationship is weak, it justifies the need for a multivariate model that combines popularity with genre, language, and timing signals.

**v2 (2026-05-23) — work-aggregated popularity:** Previously this notebook used the edition-level `gr_ratings_count` from `training_dataset.csv`. That under-counted total popularity for books where our matcher picked a minor edition. This version sums ratings across *all* Goodreads editions of the same `gr_work_id`, which matches what a publisher sees on a Goodreads work page.

Note: `work_ratings_count` here is the *total* count from the 2017 Goodreads snapshot, aggregated at work level. It's allowed as an EDA variable but NOT as a model feature (excluded from `X_train.csv` per project instruction about temporal leakage — the model uses `pre_cutoff_ratings_count` from the reviews dump instead).

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

REPO    = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM = REPO / "data" / "interim"

train = pd.read_csv(INTERIM / "training_dataset.csv", dtype=str, keep_default_na=False)

train["gr_ratings_count"]  = pd.to_numeric(train["gr_ratings_count"],  errors="coerce").fillna(0).astype(int)
train["sckn_appearances"]  = pd.to_numeric(train["sckn_appearances"],  errors="coerce").fillna(0).astype(int)
train["sckn_best_rank"]    = pd.to_numeric(train["sckn_best_rank"],    errors="coerce")
train["is_positive"]       = (train["sckn_appearances"] >= 1).astype(int)

print(f"Training cohort: {len(train):,} rows")
print(f"  SCKN positives: {train['is_positive'].sum():,} ({train['is_positive'].mean():.1%})")

Training cohort: 26,090 rows
  SCKN positives: 1,405 (5.4%)


## 0 — Compute work-aggregated ratings count

Load `goodreads_book_details.json` (book_id → [norm_title, ratings_count, pub_year, language_code, work_id]) and `goodreads_work_to_books.json`, then for each training row sum ratings across all editions of its work.

We use this `work_ratings_count` for everything below — that's the metric the supervisor's question is naturally about (and the one a publisher actually sees).

In [3]:
import time
t0 = time.time()

print("Loading book_details.json …", flush=True)
with open(INTERIM / "goodreads_book_details.json", encoding="utf-8") as f:
    book_details = json.load(f)
print(f"  {len(book_details):,} books  ({time.time()-t0:.1f}s)")

print("Loading work_to_books.json …", flush=True)
with open(INTERIM / "goodreads_work_to_books.json", encoding="utf-8") as f:
    work_to_books = json.load(f)
print(f"  {len(work_to_books):,} works  ({time.time()-t0:.1f}s)")

# Sum ratings across editions per work
print("Computing work-aggregated ratings counts …", flush=True)
work_ratings: dict[str, int] = {}
for wid, bids in work_to_books.items():
    total = 0
    for bid in bids:
        d = book_details.get(bid)
        if d:
            total += int(d[1] or 0)   # d = [norm_title, ratings_count, ...]
    work_ratings[wid] = total
print(f"  {len(work_ratings):,} works aggregated  ({time.time()-t0:.1f}s)")

# Project onto training rows. Rows without a gr_work_id fall back to their
# edition-level gr_ratings_count.
def lookup_work_ratings(row):
    wid = row.get("gr_work_id", "").strip()
    if wid and wid in work_ratings:
        return work_ratings[wid]
    return row["gr_ratings_count"]

train["work_ratings_count"] = train.apply(lookup_work_ratings, axis=1).astype(int)

print()
print("Edition-level vs work-aggregated ratings count:")
print("  edition (median / max):",
      int(train['gr_ratings_count'].median()), "/", train['gr_ratings_count'].max())
print("  work    (median / max):",
      int(train['work_ratings_count'].median()), "/", train['work_ratings_count'].max())
ratio = train['work_ratings_count'].sum() / max(train['gr_ratings_count'].sum(), 1)
print(f"  total ratio (sum work / sum edition): {ratio:.2f}x")

Loading book_details.json …
  2,360,655 books  (5.2s)
Loading work_to_books.json …
  1,521,962 works  (7.3s)
Computing work-aggregated ratings counts …
  1,521,962 works aggregated  (10.2s)

Edition-level vs work-aggregated ratings count:
  edition (median / max): 175 / 2099680
  work    (median / max): 1112 / 5064668
  total ratio (sum work / sum edition): 4.81x


## 1 — Correlation: English popularity ↔ Czech bestseller status

Both Pearson (linear) and Spearman (rank-based). Spearman is the honest number for heavy-tailed counts.

In [4]:
pearson  = train[["work_ratings_count", "sckn_appearances", "is_positive"]].corr(method="pearson")
spearman = train[["work_ratings_count", "sckn_appearances", "is_positive"]].corr(method="spearman")

print("Pearson correlations (linear):")
print(pearson.round(4).to_string())
print()
print("Spearman correlations (rank — more appropriate for heavy-tailed counts):")
print(spearman.round(4).to_string())

Pearson correlations (linear):
                    work_ratings_count  sckn_appearances  is_positive
work_ratings_count              1.0000            0.2709       0.1759
sckn_appearances                0.2709            1.0000       0.3723
is_positive                     0.1759            0.3723       1.0000

Spearman correlations (rank — more appropriate for heavy-tailed counts):
                    work_ratings_count  sckn_appearances  is_positive
work_ratings_count              1.0000            0.1515       0.1501
sckn_appearances                0.1515            1.0000       0.9995
is_positive                     0.1501            0.9995       1.0000


## 2 — Top-N analysis: does English popularity predict Czech success?

Sort training set by `work_ratings_count` descending. For each top-N bucket, compute the fraction that are SCKN positive. Baseline ≈ 5.4%.

In [5]:
sorted_by_gr = train.sort_values("work_ratings_count", ascending=False).reset_index(drop=True)
baseline = train["is_positive"].mean()

results = []
for n in [50, 100, 250, 500, 1000, 2500, 5000, 10000, len(train)]:
    top_n = sorted_by_gr.head(n)
    pos_count = top_n["is_positive"].sum()
    pos_rate = top_n["is_positive"].mean()
    lift = pos_rate / baseline
    results.append({
        "top_N":              n,
        "min_work_ratings":   int(top_n["work_ratings_count"].min()),
        "n_SCKN_positive":    int(pos_count),
        "pct_positive":       f"{pos_rate:.1%}",
        "lift_vs_baseline":   f"{lift:.2f}x",
    })

topN_table = pd.DataFrame(results)
print(f"Baseline positive rate (all training): {baseline:.1%}")
print()
print(topN_table.to_string(index=False))

Baseline positive rate (all training): 5.4%

 top_N  min_work_ratings  n_SCKN_positive pct_positive lift_vs_baseline
    50           1099287               33        66.0%           12.26x
   100            587434               51        51.0%            9.47x
   250            309098               93        37.2%            6.91x
   500            152208              146        29.2%            5.42x
  1000             78044              229        22.9%            4.25x
  2500             28205              395        15.8%            2.93x
  5000             10262              623        12.5%            2.31x
 10000              2477              910         9.1%            1.69x
 26090                10             1405         5.4%            1.00x


## 3 — Reverse view: where do SCKN positives sit on the Goodreads popularity scale?

Among the SCKN positives, distribution of `work_ratings_count` and bucket breakdown.

In [6]:
positives = train[train["is_positive"] == 1].copy()
print(f"SCKN positives: {len(positives):,}")
print()
print("work_ratings_count distribution among SCKN positives:")
print(positives["work_ratings_count"].describe(percentiles=[.1, .25, .5, .75, .9, .95]).round(0).to_string())

SCKN positives: 1,405

work_ratings_count distribution among SCKN positives:
count       1405.0
mean       95069.0
std       351565.0
min           10.0
10%          185.0
25%         1000.0
50%         6919.0
75%        35676.0
90%       161672.0
95%       443079.0
max      5064668.0


In [7]:
buckets = [
    (0,     100,     "obscure (<100 ratings)"),
    (100,   1000,    "niche (100–999)"),
    (1000,  10000,   "recognized (1k–10k)"),
    (10000, 100000,  "popular (10k–100k)"),
    (100000, 10**8,  "viral (100k+)"),
]

print("SCKN positives by Goodreads popularity bucket (work-aggregated):")
for lo, hi, label in buckets:
    mask = (positives["work_ratings_count"] >= lo) & (positives["work_ratings_count"] < hi)
    n = mask.sum()
    pct = n / len(positives) * 100
    bar = "█" * int(pct / 2)
    print(f"  {label:<28} : {n:>4} ({pct:>5.1f}%)  {bar}")

SCKN positives by Goodreads popularity bucket (work-aggregated):
  obscure (<100 ratings)       :   88 (  6.3%)  ███
  niche (100–999)              :  263 ( 18.7%)  █████████
  recognized (1k–10k)          :  423 ( 30.1%)  ███████████████
  popular (10k–100k)           :  437 ( 31.1%)  ███████████████
  viral (100k+)                :  194 ( 13.8%)  ██████


## 4 — Concrete examples

### 4a — English bestsellers that flopped in Czech

In [8]:
english_hits_czech_flops = (
    train[train["is_positive"] == 0]
    .sort_values("work_ratings_count", ascending=False)
    .head(20)
    [["author", "original_title", "czech_title", "czech_pub_year",
      "source_lang", "work_ratings_count", "sckn_appearances"]]
)
print("TOP 20 high-Goodreads-popularity books (work-aggregated) that were NEVER in SCKN top 10:")
english_hits_czech_flops

TOP 20 high-Goodreads-popularity books (work-aggregated) that were NEVER in SCKN top 10:


,author,original_title,czech_title,czech_pub_year,source_lang,work_ratings_count,sckn_appearances
7562,"Lee, Harper",To kill a mockingbird,Jako zabít ptáčka,2009,eng,3399207,0
6516,"Fitzgerald, Francis Scott",Great Gatsby,Velký Gatsby,2008,eng,2844050,0
456,"Orwell, George",Nineteen eighty-four,1984,2003,eng,2118765,0
5842,"Orwell, George",Animal farm,Animal farm =,2008,eng,2030347,0
3806,"Tolkien, J. R. R",Fellowship of the ring,Pán prstenů,2006,eng,1876076,0
159,"Golding, William",Lord of the flies,Pán much,2003,eng,1704682,0
1652,"Shakespeare, William",Romeo and Juliet,Romeo a Julie,2004,eng,1693113,0
522,"Sebold, Alice",Lovely bones,Pevné pouto,2003,eng,1684392,0
1974,"Steinbeck, John",Of mice and men,O myších a lidech,2004,eng,1546521,0
7893,"Riordan, Rick",Lightning thief,Percy Jackson,2009,eng,1457494,0


### 4b — Czech bestsellers with low Goodreads popularity (work-aggregated)

After work-aggregation, this list should contain *fewer* artifacts (non-English originals where we'd matched to a minor edition). Books that remain here are real "low-Goodreads-exposure but Czech bestseller" cases.

In [9]:
czech_hits_low_english = (
    positives
    .sort_values("work_ratings_count", ascending=True)
    .head(20)
    [["author", "original_title", "czech_title", "czech_pub_year",
      "source_lang", "work_ratings_count", "sckn_appearances", "sckn_best_rank"]]
)
print("TOP 20 SCKN-positive books with the LOWEST work-aggregated Goodreads ratings count:")
czech_hits_low_english

TOP 20 SCKN-positive books with the LOWEST work-aggregated Goodreads ratings count:


,author,original_title,czech_title,czech_pub_year,source_lang,work_ratings_count,sckn_appearances,sckn_best_rank
23286,"Garnier, Stéphane",Agir et penser comme un chat,Chovejte se jako kočka,2018,fre,10,16,2.0
18060,Áslaug Jónsdóttir,Nei! sagði litla skrímslið,Ne! řeklo strašidýlko,2016,ice,10,1,7.0
18227,"Bass, Guy",Beast of Grubbers Nubbin,Záplaťák,2016,eng,10,1,2.0
165,"Bittleston, Jennie",Secrets of yoga,Tajemství jógy,2003,eng,12,1,10.0
22807,"Martin, George R. R",Fire and blood,Oheň a krev,2019,eng,12,4,2.0
16684,"Peschke, M",Summer camp queen,Káťa Líbezná,2015,eng,12,1,9.0
1935,"Friedrich, Joachim",4 1/2 Freunde,Čtyři a půl kamaráda,2004,ger,13,1,8.0
2065,"Fulghum, Robert",Third wish,Třetí přání,2004,eng,14,18,1.0
766,"Andersen, Hans Christian",Eventyr,Pohádky,2012,hun|dan,14,1,4.0
19191,"Kaiblinger, Sonja",Totgesagte leben länger,Smrťák Harry,2017,ger,14,1,5.0


### 4c — Both ends together (the "obvious matches")

In [10]:
obvious = (
    positives
    .sort_values("work_ratings_count", ascending=False)
    .head(20)
    [["author", "original_title", "czech_title", "czech_pub_year",
      "source_lang", "work_ratings_count", "sckn_appearances", "sckn_best_rank"]]
)
print("TOP 20 SCKN-positive books with the HIGHEST work-aggregated Goodreads ratings:")
obvious

TOP 20 SCKN-positive books with the HIGHEST work-aggregated Goodreads ratings:


,author,original_title,czech_title,czech_pub_year,source_lang,work_ratings_count,sckn_appearances,sckn_best_rank
9204,"Collins, Suzanne",Hunger games,Hunger games,2010,eng,5064668,5,2.0
1013,"Rowling, J. K",Harry Potter and the philosopher's stone,Harry Potter a kámen mudrců,2017,eng,4970387,70,1.0
3221,"Meyer, Stephenie",Twilight,Stmívání,2008,eng,3991256,57,1.0
12773,"Green, John",Fault in our stars,Hvězdy nám nepřály,2014,eng,2563830,16,3.0
12190,"Roth, Veronica",Divergent,Divergence,2012,eng,2277480,14,2.0
923,"Austen, Jane",Pride and prejudice,Pýcha a předsudek,2003,eng,2226979,1,5.0
530,"Tolkien, J. R. R",Hobbit,"Hobit, aneb, Cesta tam a zase zpátky",2003,eng,2215773,1,10.0
3589,"Salinger, J. D",Catcher in the rye,Kdo chytá v žitě,2010,eng,2163583,1,10.0
324,"Brown, Dan",Angels & demons,Andělé a démoni,2003,eng,2123290,42,3.0
1603,"Frank, Anne",Achterhuis,Deník,2006,ger|dut,2075813,1,8.0


## 5 — Conditional probability tables

P(SCKN+ | work_ratings ≥ T) at various thresholds.

In [11]:
thresholds = [100, 500, 1000, 5000, 10000, 50000, 100000, 500000, 1_000_000]

rows = []
for T in thresholds:
    mask = train["work_ratings_count"] >= T
    n_total = mask.sum()
    n_pos   = train.loc[mask, "is_positive"].sum()
    p_cond  = n_pos / n_total if n_total > 0 else 0
    lift    = p_cond / baseline if baseline > 0 else 0
    rows.append({
        "work_ratings ≥":   f"{T:>9,}",
        "n_books_in_pool":  n_total,
        "n_SCKN_positive":  int(n_pos),
        "P(SCKN+ | pool)":  f"{p_cond:.1%}",
        "lift_vs_baseline": f"{lift:.2f}x",
    })

print(f"Baseline P(SCKN+) on whole training: {baseline:.1%}")
print()
print(pd.DataFrame(rows).to_string(index=False))

Baseline P(SCKN+) on whole training: 5.4%

work_ratings ≥  n_books_in_pool  n_SCKN_positive P(SCKN+ | pool) lift_vs_baseline
           100            21489             1317            6.1%            1.14x
           500            16043             1141            7.1%            1.32x
         1,000            13447             1054            7.8%            1.46x
         5,000             7401              774           10.5%            1.94x
        10,000             5094              631           12.4%            2.30x
        50,000             1526              296           19.4%            3.60x
       100,000              790              194           24.6%            4.56x
       500,000              139               65           46.8%            8.68x
     1,000,000               54               35           64.8%           12.04x


## 6 — Summary for thesis discussion (work-aggregated)

Fill in from above:

1. **Correlation work_ratings_count ↔ sckn_appearances (Spearman):** ___
2. **Top-100 by work_ratings_count → % SCKN positive:** ___%  (lift ___x)
3. **% of SCKN positives that are "obscure" (< 1000 work ratings):** ___%
4. **Concrete example, English bestseller that flopped in Czech:** ___
5. **Concrete example, Czech bestseller with low Goodreads count:** ___

**Pattern check:** if (2) is < 50% and (3) is meaningfully positive (> 5%), the data still supports the thesis framing — English popularity alone is a weak filter, multivariate model justified.

These numbers go into:
- Reply to supervisor (Section: EDA support for the framing).
- Thesis Chapter on data analysis.
- Thesis Discussion (justifying why we need a multivariate model).